# Final XGBoost feature selection

This notebook tests conservative feature pruning using the existing XGBoost feature importance, SHAP, correlation, and VIF diagnostics. The original M1–M4 models remain unchanged.

Three earlier candidates and four matched correlation-reduced variants are retrained:

- **Accuracy-first:** M1 without `month`.
- **Research-preserving:** M4 without `rainfall_hourly_mm` and `indoor_humidity_pct_last_week`.
- **Correlation-reduced:** M4 without `floor`, `month`, and `is_weekend`, following the supplied feature-selection notebook.
- **Matched ablations:** `correlation_reduced_M1` through `correlation_reduced_M4`, each original M1–M4 set with those exclusions applied.

A feature is recommended for removal only when the retrained candidate is no worse on the common scored operating-hours test rows.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT=Path.cwd()
if ROOT.name!='4-XGBoost':
    ROOT=next(p for p in [ROOT,*ROOT.parents] if p.name=='4-XGBoost')
OUTPUT=ROOT/'outputs'
FINAL=OUTPUT/'final_models'
with open(FINAL/'feature_definitions.json') as f:
    definitions=json.load(f)
for name, details in definitions['definitions'].items():
    print(name, '| base:', details['base'], '| dropped:', ', '.join(details['dropped']))
    print('retained:', ', '.join(definitions['candidates'][name]))
    print()

accuracy_first | base: M1 | dropped: month
retained: floor, zone_code, hour, weekday, is_weekend, is_holiday, ac_load_kw_yesterday, ac_load_kw_2_days_ago, ac_load_kw_last_week

research_preserving | base: M4 | dropped: rainfall_hourly_mm, indoor_humidity_pct_last_week
retained: floor, zone_code, hour, weekday, month, is_weekend, is_holiday, ac_load_kw_yesterday, ac_load_kw_2_days_ago, ac_load_kw_last_week, indoor_temp_c_yesterday, indoor_temp_c_last_week, indoor_temp_c_yesterday_avg, indoor_humidity_pct_yesterday, indoor_humidity_pct_yesterday_avg, outdoor_temp_hourly_c, outdoor_temp_daily_avg_c, outdoor_humidity_hourly_pct, outdoor_humidity_daily_avg_pct, rainfall_daily_total_mm, solar_radiation_hourly_wm2, solar_radiation_daily_total_mjm2

correlation_reduced | base: M4 | dropped: floor, month, is_weekend
retained: zone_code, hour, weekday, is_holiday, ac_load_kw_yesterday, ac_load_kw_2_days_ago, ac_load_kw_last_week, indoor_temp_c_yesterday, indoor_temp_c_last_week, indoor_temp_c_ye

## 1. Why these variables were considered

The supplied feature-selection notebook retained 21 predictors and excluded three: `floor`, `month`, and `is_weekend`. The exclusions are based on near-duplicate information: `floor` is almost determined by `zone_code`, `is_weekend` is determined by `weekday`, and validation months (April, August, and December) do not occur in the training split. The correlation-reduced candidate applies these exclusions to M4 while retaining the indoor and weather variables for the research question.

The earlier candidates are retained for comparison. Correlation-based exclusions are tested by retraining rather than assumed to improve XGBoost, because correlated variables can still provide useful split structure.


In [2]:
importance=pd.concat([
    pd.read_csv(FINAL/f'{model}/feature_importance.csv').assign(model=model)
    for model in definitions['definitions']
],ignore_index=True)
for model in importance.model.unique():
    print(model)
    display(importance[importance.model.eq(model)].sort_values('gain',ascending=False).head(15))


accuracy_first


,feature,gain,model
0,ac_load_kw_last_week,12615.795898,accuracy_first
1,is_holiday,11758.485352,accuracy_first
2,is_weekend,4035.730225,accuracy_first
3,ac_load_kw_yesterday,3250.302246,accuracy_first
4,weekday,1464.544922,accuracy_first
5,hour,1317.722412,accuracy_first
6,zone_code,288.546478,accuracy_first
7,ac_load_kw_2_days_ago,269.744080,accuracy_first
8,floor,213.256714,accuracy_first


research_preserving


,feature,gain,model
9,ac_load_kw_last_week,7148.612305,research_preserving
10,is_holiday,6230.617188,research_preserving
11,is_weekend,2680.698975,research_preserving
12,ac_load_kw_yesterday,2251.515625,research_preserving
13,weekday,1285.102173,research_preserving
14,solar_radiation_hourly_wm2,372.612122,research_preserving
15,indoor_temp_c_last_week,313.802307,research_preserving
16,hour,309.677094,research_preserving
17,zone_code,222.132751,research_preserving
18,indoor_temp_c_yesterday_avg,169.344147,research_preserving


correlation_reduced


,feature,gain,model
31,ac_load_kw_last_week,7660.533203,correlation_reduced
32,is_holiday,6307.128906,correlation_reduced
33,ac_load_kw_yesterday,2267.658691,correlation_reduced
34,weekday,1757.043701,correlation_reduced
35,indoor_temp_c_last_week,1095.425903,correlation_reduced
36,ac_load_kw_2_days_ago,542.481445,correlation_reduced
37,hour,509.580994,correlation_reduced
38,solar_radiation_hourly_wm2,491.868591,correlation_reduced
39,zone_code,267.610840,correlation_reduced
40,outdoor_temp_daily_avg_c,186.897720,correlation_reduced


... (4 outputs left out)

correlation_reduced_M3


,feature,gain,model
72,ac_load_kw_last_week,20642.662109,correlation_reduced_M3
73,is_holiday,14701.507812,correlation_reduced_M3
74,ac_load_kw_yesterday,3537.931396,correlation_reduced_M3
75,weekday,3172.536621,correlation_reduced_M3
76,solar_radiation_hourly_wm2,1138.440308,correlation_reduced_M3
77,hour,548.022400,correlation_reduced_M3
78,outdoor_temp_daily_avg_c,515.923096,correlation_reduced_M3
79,ac_load_kw_2_days_ago,343.280914,correlation_reduced_M3
80,zone_code,335.897797,correlation_reduced_M3
81,rainfall_daily_total_mm,320.240601,correlation_reduced_M3


correlation_reduced_M4


,feature,gain,model
87,ac_load_kw_last_week,7660.533203,correlation_reduced_M4
88,is_holiday,6307.128906,correlation_reduced_M4
89,ac_load_kw_yesterday,2267.658691,correlation_reduced_M4
90,weekday,1757.043701,correlation_reduced_M4
91,indoor_temp_c_last_week,1095.425903,correlation_reduced_M4
92,ac_load_kw_2_days_ago,542.481445,correlation_reduced_M4
93,hour,509.580994,correlation_reduced_M4
94,solar_radiation_hourly_wm2,491.868591,correlation_reduced_M4
95,zone_code,267.610840,correlation_reduced_M4
96,outdoor_temp_daily_avg_c,186.897720,correlation_reduced_M4


## 2. Common scored test metrics

These results use the same `zone_id + timestamp` rows where the baseline file has `scored == 1`. Lower values are better.

In [3]:
reduced=pd.read_csv(FINAL/'common_metrics.csv').rename(columns={'slice':'slice'})
reduced['slice']=reduced['slice'].replace({'operating_hours':'operating hours','all_hours':'all hours'})
original=pd.read_csv(OUTPUT/'model_comparison/common_model_metrics.csv').rename(columns={'hours':'slice'})
comparison=pd.concat([
    original[['model','slice','n','MAE','RMSE','CV(RMSE)%','NMBE%','WAPE%']],
    reduced.rename(columns={'mae':'MAE','rmse':'RMSE','cv_rmse':'CV(RMSE)%','nmbe':'NMBE%','wape':'WAPE%'})[['model','slice','n','MAE','RMSE','CV(RMSE)%','NMBE%','WAPE%']]
],ignore_index=True)
display(comparison[comparison.slice.eq('operating_hours')].round(4))

,model,slice,n,MAE,RMSE,CV(RMSE)%,NMBE%,WAPE%


## 3. Recommendation

The recommendation is based on retrained common scored operating-hours performance. The accuracy-first and research-preserving candidates remain available, while the correlation-reduced candidate directly tests the supplied feature-selection exclusions.


In [4]:
operating=comparison[comparison.slice.eq('operating hours')].set_index('model')
base=operating.loc['accuracy_first']
for model in ['accuracy_first','research_preserving','correlation_reduced']:
    row=operating.loc[model]
    print(model)
    for metric in ['MAE','RMSE','CV(RMSE)%','WAPE%']:
        print(f'  {metric}: {row[metric]:.4f} (change vs accuracy_first: {row[metric]-base[metric]:+.4f})')
best=operating['WAPE%'].idxmin()
print(f'\nBest operating-hours WAPE in this run: {best}')
print('Supplied exclusions tested: floor, month, is_weekend')
print('Do not drop a correlated feature without checking retrained test performance.')


accuracy_first
  MAE: 1.4788 (change vs accuracy_first: +0.0000)
  RMSE: 2.4670 (change vs accuracy_first: +0.0000)
  CV(RMSE)%: 0.1804 (change vs accuracy_first: +0.0000)
  WAPE%: 0.1081 (change vs accuracy_first: +0.0000)
research_preserving
  MAE: 1.7598 (change vs accuracy_first: +0.2809)
  RMSE: 2.7441 (change vs accuracy_first: +0.2772)
  CV(RMSE)%: 0.2007 (change vs accuracy_first: +0.0203)
  WAPE%: 0.1287 (change vs accuracy_first: +0.0205)
correlation_reduced
  MAE: 1.7822 (change vs accuracy_first: +0.3034)
  RMSE: 2.8029 (change vs accuracy_first: +0.3360)
  CV(RMSE)%: 0.2050 (change vs accuracy_first: +0.0246)
  WAPE%: 0.1303 (change vs accuracy_first: +0.0222)

Best operating-hours WAPE in this run: accuracy_first
Supplied exclusions tested: floor, month, is_weekend
Do not drop a correlated feature without checking retrained test performance.


## 4. Retained and dropped variables

### Accuracy-first final model

Drop only `month` from M1. Retain AC lags, operating/calendar variables, `zone_code`, and `floor`.

### Research-preserving model

Drop `rainfall_hourly_mm` and `indoor_humidity_pct_last_week`, but retain the remaining indoor and weather groups.

### Correlation-reduced model

Drop `floor`, `month`, and `is_weekend` from M4. This leaves the 21 predictors marked as kept by the supplied feature-selection notebook and removes the three marked as excluded.

The final deployment choice should be made from the retrained metrics below, not from correlation or feature importance alone.


## Matched M1–M4 correlation ablation

This table compares each original M1–M4 model with its matched variant that removes `floor`, `month`, and `is_weekend`. The models were independently retuned and evaluated on the same common scored operating-hours rows. Lower values are better.


In [5]:
orig = pd.read_csv(OUTPUT / 'evaluation' / 'common_scored_metrics.csv')
orig = orig[orig['slice'].isin(['all_hours', 'operating_hours'])].copy()
orig['model'] = orig['model'].map(lambda x: x if x in ['M1','M2','M3','M4'] else x)
reduced = pd.read_csv(FINAL / 'common_metrics.csv')
reduced = reduced[reduced['model'].str.startswith('correlation_reduced_M')].copy()
reduced['base_model'] = reduced['model'].str.replace('correlation_reduced_', '', regex=False)
orig['base_model'] = orig['model']
orig['variant'] = 'original'
reduced['variant'] = 'correlation_reduced'
matched = pd.concat([orig[['base_model','variant','slice','n','mae','rmse','cv_rmse','nmbe','wape']], reduced[['base_model','variant','slice','n','mae','rmse','cv_rmse','nmbe','wape']]], ignore_index=True)
display(matched[matched['slice'].eq('operating_hours')].sort_values(['base_model','variant']).round(4))
wide = matched[matched['slice'].eq('operating_hours')].pivot(index='base_model', columns='variant', values=['mae','rmse','cv_rmse','nmbe','wape'])
display(wide.round(4))


,base_model,variant,slice,n,mae,rmse,cv_rmse,nmbe,wape
9,M1,correlation_reduced,operating_hours,12602.0,1.4960,2.4766,0.1811,0.0221,0.1094
1,M1,original,operating_hours,12602.0,1.4995,2.4820,0.1815,0.0160,0.1097
11,M2,correlation_reduced,operating_hours,12602.0,1.8451,2.9967,0.2192,0.0662,0.1349
3,M2,original,operating_hours,12602.0,1.9263,3.1497,0.2303,0.0792,0.1409
13,M3,correlation_reduced,operating_hours,12602.0,1.7652,2.7852,0.2037,-0.0332,0.1291
5,M3,original,operating_hours,12602.0,1.6165,2.6516,0.1939,-0.0237,0.1182
15,M4,correlation_reduced,operating_hours,12602.0,1.7822,2.8029,0.2050,0.0219,0.1303
7,M4,original,operating_hours,12602.0,2.0100,3.1217,0.2283,0.0337,0.1470


mae                         rmse           \
variant    correlation_reduced original correlation_reduced original   
base_model                                                             
M1                      1.4960   1.4995              2.4766   2.4820   
M2                      1.8451   1.9263              2.9967   3.1497   
M3                      1.7652   1.6165              2.7852   2.6516   
M4                      1.7822   2.0100              2.8029   3.1217   

                       cv_rmse                         nmbe           \
variant    correlation_reduced original correlation_reduced original   
base_model                                                             
M1                      0.1811   0.1815              0.0221   0.0160   
M2                      0.2192   0.2303              0.0662   0.0792   
M3                      0.2037   0.1939             -0.0332  -0.0237   
M4                      0.2050   0.2283              0.0219   0.0337   

                          wape           
variant    correlation_reduced original  
base_model                               
M1                      0.1094   0.1097  
M2                      0.1349   0.1409  
M3                      0.1291   0.1182  
M4                      0.1303   0.1470

### Interpretation

In this run, removing the three selected features did not improve operating-hours WAPE for any original M1–M4 configuration. The reduced M1 and M3 variants were close to their originals, while the reduced M2 and M4 variants were worse. Correlation reduction should therefore be treated as an interpretability/ablation experiment, not an automatic accuracy improvement.


In [6]:
print('Final model artifacts:')
for path in sorted(FINAL.rglob('*')):
    if path.is_file(): print(path.relative_to(FINAL))

Final model artifacts:
accuracy_first/feature_importance.csv
accuracy_first/metadata.json
accuracy_first/model.json
accuracy_first/predictions.csv
accuracy_first/shap_bar.png
accuracy_first/shap_summary.png
common_metrics.csv
correlation_reduced/feature_importance.csv
correlation_reduced/metadata.json
correlation_reduced/model.json
correlation_reduced/predictions.csv
correlation_reduced/shap_bar.png
correlation_reduced/shap_summary.png
correlation_reduced_M1/feature_importance.csv
correlation_reduced_M1/metadata.json
correlation_reduced_M1/model.json
correlation_reduced_M1/predictions.csv
correlation_reduced_M1/shap_bar.png
correlation_reduced_M1/shap_summary.png
correlation_reduced_M2/feature_importance.csv
correlation_reduced_M2/metadata.json
correlation_reduced_M2/model.json
correlation_reduced_M2/predictions.csv
correlation_reduced_M2/shap_bar.png
correlation_reduced_M2/shap_summary.png
correlation_reduced_M3/feature_importance.csv
correlation_reduced_M3/metadata.json
correlation_r